# DeBERTa-v3-base PCL Detection (Trainer API, Prompt Notebook Rewrite)

This notebook rewrites the training pipeline into a simple Hugging Face Trainer workflow using:
- `AutoModelForSequenceClassification`
- `Trainer`
- `TrainingArguments`

Model selection tracks **PCL-class F1** (`f1_pcl`) and training uses a weighted random sampler.


In [ ]:
!pip install contractions huggingface_hub


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login

login(token=hf_token)


In [ ]:
import os
import re
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import WeightedRandomSampler

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns
import contractions

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 256
NUM_LABELS = 2

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
SAMPLER_POWER = 1.0  # 1.0=1/k (balanced), 0.5=1/sqrt(k)

EVAL_EVERY = 20
LOG_EVERY = 50

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection'
else:
    DATA_ROOT = '.'

TSV_PATH = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')

OUTPUT_DIR = 'checkpoints/deberta_v3_prompt_trainer'
LOG_DIR = 'logs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DATA_ROOT   : {DATA_ROOT}')
print(f'TSV_PATH    : {TSV_PATH}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')


In [ ]:
def load_task1(train_path: str) -> pd.DataFrame:
    """
    Load Task 1 data and convert original labels to binary:
      0/1 -> 0 (No-PCL)
      2/3/4 -> 1 (PCL)
    """
    rows = []
    with open(train_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue

            par_id = parts[0]
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]
            text = parts[4]
            orig_label = parts[-1]
            label = 0 if orig_label in {'0', '1'} else 1

            rows.append(
                {
                    'par_id': str(par_id),
                    'art_id': art_id,
                    'keyword': keyword,
                    'country': country,
                    'text': text,
                    'label': label,
                    'orig_label': orig_label,
                }
            )

    return pd.DataFrame(
        rows,
        columns=['par_id', 'art_id', 'keyword', 'country', 'text', 'label', 'orig_label'],
    )


def preprocess_text(text: str) -> str:
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)

print(f'Loaded dataset: {len(df):,} samples')
print(df['label'].value_counts().sort_index().rename({0: 'No-PCL', 1: 'PCL'}))


In [ ]:
# ============================================================
# Official Train/Dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df = pd.read_csv(DEV_IDS_PATH, dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)
dev_df = df[df['par_id'].isin(dev_par_ids)].copy().reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].copy().reset_index(drop=True)
if len(leftover_df) > 0:
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)
    print(f'Appended {len(leftover_df):,} unassigned samples to training set.')


def describe_split(name: str, frame: pd.DataFrame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum())
    n_no_pcl = int((frame['label'] == 0).sum())
    ratio = f'1:{(n_no_pcl / n_pcl):.1f}' if n_pcl > 0 else 'undefined'
    print(f'{name:<6} -> total={n:,} | PCL={n_pcl:,} | No-PCL={n_no_pcl:,} | ratio={ratio}')


describe_split('Train', train_df)
describe_split('Dev', dev_df)


In [ ]:
# ============================================================
# Tokenization + Hugging Face Datasets
# ============================================================
train_hf = Dataset.from_pandas(
    train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)
dev_hf = Dataset.from_pandas(
    dev_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


train_ds = train_hf.map(tokenize, batched=True, remove_columns=['text'])
dev_ds = dev_hf.map(tokenize, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
dev_ds = dev_ds.rename_column('label', 'labels')

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Weighted random sampler weights
n_train = len(train_df)
n_pcl = int((train_df['label'] == 1).sum())
n_no_pcl = int((train_df['label'] == 0).sum())
k_p = n_pcl / n_train
k_n = n_no_pcl / n_train

w_pcl = 1.0 / (k_p ** SAMPLER_POWER)
w_no_pcl = 1.0 / (k_n ** SAMPLER_POWER)
sample_weights = np.where(train_df['label'].values == 1, w_pcl, w_no_pcl).astype(np.float64)
expected_pos_rate = (n_pcl * w_pcl) / ((n_pcl * w_pcl) + (n_no_pcl * w_no_pcl))

print(f'SAMPLER_POWER         : {SAMPLER_POWER:.2f}')
print(f'Sampling weights      : PCL={w_pcl:.4f} | No-PCL={w_no_pcl:.4f}')
print(f'Expected sampled PCL% : {expected_pos_rate*100:.2f}%')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(dev_ds)


In [ ]:
# ============================================================
# Metrics
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_no_pcl': f1_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'f1_pcl': f1_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'prec_no_pcl': precision_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'prec_pcl': precision_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'rec_no_pcl': recall_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'rec_pcl': recall_score(labels, preds, pos_label=1, average='binary', zero_division=0),
    }


In [ ]:
# ============================================================
# Model + TrainingArguments + WeightedSamplerTrainer
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

args_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': NUM_EPOCHS,
    'per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'lr_scheduler_type': 'cosine',
    'logging_steps': LOG_EVERY,
    'save_strategy': 'steps',
    'save_steps': EVAL_EVERY,
    'eval_steps': EVAL_EVERY,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_pcl',
    'greater_is_better': True,
    'save_total_limit': 2,
    'report_to': 'none',
    'seed': SEED,
    'fp16': torch.cuda.is_available(),
}

# Handle transformers API change: evaluation_strategy -> eval_strategy
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in ta_sig:
    args_kwargs['eval_strategy'] = 'steps'
else:
    args_kwargs['evaluation_strategy'] = 'steps'

training_args = TrainingArguments(**args_kwargs)


class WeightedSamplerTrainer(Trainer):
    def __init__(self, *args, sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights = sample_weights

    def _get_train_sampler(self):
        if self.sample_weights is None:
            return super()._get_train_sampler()
        if self.train_dataset is None:
            return None
        return WeightedRandomSampler(
            weights=torch.as_tensor(self.sample_weights, dtype=torch.double),
            num_samples=len(self.sample_weights),
            replacement=True,
        )


trainer = WeightedSamplerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    sample_weights=sample_weights,
)

print('Trainer configured.')


In [ ]:
train_result = trainer.train()
train_result


In [ ]:
eval_metrics = trainer.evaluate()
print('Evaluation metrics:')
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')


In [ ]:
# ============================================================
# Detailed evaluation report
# ============================================================
pred_output = trainer.predict(dev_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred No-PCL', 'Pred PCL'],
    yticklabels=['True No-PCL', 'True PCL'],
)
plt.title('Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()


In [ ]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, 'best')
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f'Saved best model and tokenizer to: {BEST_MODEL_DIR}')
